In [1]:
# Ensure the following are installed via command line in environment before running

# remotes::install_github("jokergoo/circlize@9b21578")
# remotes::install_github("jokergoo/ComplexHeatmap@7d95ca5")
# remotes::install_github("immunogenomics/presto@31dc97f")
# remotes::install_github("jinworks/CellChat@88c2e13")
# BiocManager::install("GenomeInfoDbData")

In [1]:
suppressPackageStartupMessages({
    library(tidyverse)
    library(zellkonverter)
    library(SingleCellExperiment)
    library(CellChat)
})

In [2]:
dist_out_dir <- "/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance"
dist_out_dir_cellchat <- "/home/workspace/spatial_mouse_lung_outputs/downstream_analysis/distance/cellchat_tls-category"

In [3]:
sce = readH5AD(file.path(dist_out_dir_cellchat, "adata_cellchat_prepped.h5ad"))

Warning message:
“The names of these selected uns items have been modified to match R
conventions: '_scvi_manager_uuid' -> 'X_scvi_manager_uuid' and '_scvi_uuid' ->
'X_scvi_uuid'”
Warning message:
“The names of these selected obs columns have been modified to match R
conventions: '_scvi_batch' -> 'X_scvi_batch' and '_scvi_labels' ->
'X_scvi_labels'”


In [4]:
#sce$Timepoint <- stringr::str_extract(sce$batch, "\\d+")
sce$sample_label <- factor(sce$sample_label, levels = c("HDM_day3", "HDM_day30"))

## Run cellchat

In [8]:
run_cellchat <- function(sce_tmp, sample_label_select, out_dir, trim = 0.05, interaction_range = 100, contact_range = 10, population_size = FALSE) {
    
    print("step: data.input"); flush.console()
    data.input = assay(sce_tmp, "X") # X are the log norm counts here, see part 1
    meta = data.frame(labels = sce_tmp$label_fine,
                    samples = sce_tmp$sample_label_cp, # (KA - I only have one sample, this is just a copy of sample_label
                    row.names = colnames(sce_tmp))
    print("step: spatial.locs"); flush.console()
    # spatial.locs = reducedDim(sce_tmp, 'spatial') |> as.data.frame() 
    spatial.locs = reducedDim(sce_tmp, 'spatial')  
    scale.factors = list(spot.diameter = 5, spot = 5)
    spatial.factors = data.frame(ratio = 1, tol = 5)

    print("running: createCellChat")
    cellchat <-
        createCellChat(
            object = data.input,
            meta = meta,
            group.by = "labels",
            datatype = "spatial",
            coordinates = spatial.locs,
            spatial.factors = spatial.factors
        )


    CellChatDB <- CellChatDB.mouse # use CellChatDB.human if running on human data

    # use all CellChatDB for cell-cell communication analysis
    CellChatDB.use <- CellChatDB # simply use the default CellChatDB

    # set the used database in the object
    cellchat@DB <- CellChatDB.use

    # subset the expression data of signaling genes for saving computation cost
    cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database

    print("running: identifyOverExpressedGenes, identifyOverExpressedInteractions")
    cellchat <- identifyOverExpressedGenes(cellchat, do.DE = FALSE, min.cells = 10) 
    cellchat <- identifyOverExpressedInteractions(cellchat)

    # Typically, contact.range = 10, which is a typical human cell size
    print("running: computeCommunProb")
    cat("using interaction range: ", interaction_range)
    cat("using contact range: ", contact_range)
    cellchat <- computeCommunProb(cellchat,
        type = "truncatedMean", trim = trim, 
        distance.use = TRUE, interaction.range = interaction_range,
        scale.distance = 1, 
        contact.dependent = TRUE, 
        contact.range = contact_range, 
        population.size = population_size, 
    )
    # Filter out the cell-cell communication if there are only few number of cells in certain cell groups
    print("running: filterCommunication")
    cellchat <- filterCommunication(cellchat, min.cells = 10)

    print("running: computeCommunProbPathway")
    cellchat <- computeCommunProbPathway(cellchat)
    cellchat <- aggregateNet(cellchat)

    filename <- paste0("cellchat_", sample_label_select, "_trim_", trim, "_intrange_", interaction_range, "_contact_", contact_range, "_popsize_", population_size, ".rds")
    print(paste("running: saveRDS "))
    saveRDS(cellchat, file = file.path(out_dir, filename))
}

In [9]:
cellchat_sample <- function(sce, sample_label_select, out_dir, trim = 0.05, interaction_range=100, contact_range=10, population_size = FALSE) {

    sce_tmp = sce[,sce$sample_label == sample_label_select]
    
    sce_tmp$sample_label_cp <- sce_tmp$sample_label
    sce_tmp$sample_label_cp <- droplevels(sce_tmp$sample_label_cp)
    sce_tmp$label_fine <- droplevels(sce_tmp$label_fine)

    print("run_cellchat"); flush.console()
    run_cellchat(sce_tmp, sample_label_select, out_dir, trim, interaction_range, contact_range, population_size)
}

In [11]:
# Run on HDM day 3 
sample_label_select <- 'HDM_day3'
trim <- 0.1
interaction_range = 50
contact_range = 10
population_size <- FALSE
out_dir = dist_out_dir_cellchat
cellchat_sample(sce, sample_label_select,out_dir, trim, interaction_range, contact_range, population_size)

[1] "run_cellchat"
[1] "step: data.input"
[1] "step: spatial.locs"
[1] "running: createCellChat"
[1] "Create a CellChat object from a data matrix"
Create a CellChat object from spatial transcriptomics data... 
Set cell identities for the new CellChat object 
The cell groups used for CellChat analysis are  AT1, AT2, Alv Mf, Art, B cell, CD4 act (tls core), CD4 act (tls outer), CD4 act (tls surround), CD4 naive, CD8 act, CD8 naive, Cap, Cap-a, Ccr7+ cDC2, Ccr7- cDC2, Ciliated, Club, Col13a1+ fibroblast, Col14a1+ fibroblast, ILC2, Int Mf, Lymph, Mesothelial, Mono, Myofibroblast, NK cell, Neut, Pericyte 1, Pericyte 2, Plasmablast, SMC, Vein, cDC1, gd T cell 
[1] "running: identifyOverExpressedGenes, identifyOverExpressedInteractions"
The number of highly variable ligand-receptor pairs used for signaling inference is 209 
[1] "running: computeCommunProb"
using interaction range:  50truncatedMean is used for calculating the average gene expression per cell group. 
[1] ">>> Run CellChat on sp